# 5th Notebook. To be run after feature extraction
## Computes KL divergence of feature means across trial and inter-trial inverals from full-length mask feature vectors.

## NEED TO ADD:
- TC/PC processing differentiation. TC can use current. PC needs to be using PC trial definitions of light. Will need to use exp log to get a give video's day's trial definitions (steal logic from NB2).

## STEP 1: Define KL functions.
This block defines two mathematical approaches to computing KL divergence:

1. **kl_gaussian_diag()** - "Diagonal" or "Independent Features" approach
    This assumes that each feature varies independently (probably not true) and treats the 10-dimensional
    distribution as 10 separate 1D Gaussians. Use when features don't strongly correlate.
    Formula: KL = Σ[0.5 * (log(σ²_B/σ²_A) + (σ²_A + (μ_A - μ_B)²)/σ²_B - 1)]



2. **kl_gaussian_full()** - "Full Covariance" or "Multivariate" approach
    This version accounts for correlations between features; the 10-dimensional distribution is treated as one 
    multivariate Gaussian, which requires computing and inverting the 10x10 covariance matrix (yikes!). This 
    approach is data hungry, but should be used when features have known correlations, which is certainly the 
    case here.
    Formula: KL = 0.5 * [tr(Σ_B⁻¹Σ_A) + (μ_B-μ_A)ᵀΣ_B⁻¹(μ_B-μ_A) - d + log(det(Σ_B)/det(Σ_A))]


**Why both?**
- Diagonal is robust but may underestimate divergence if correlations differ between conditions
- Full is theoretically correct but can be numerically unstable with ~2000 samples in 10 dimensions
- Comparing both helps validate results

**Returns:**
- Total KL divergence (scalar)
- Per-feature contributions (array) - shows which features drive the divergence

# Imports

In [1]:
import numpy as np
import pandas as pd
import os
from pathlib import Path
import json
import matplotlib.pyplot as plt
import pdb
import glob
import datetime
from collections import defaultdict
import traceback

# Define session to process

In [2]:
VIDEO_PREFIX_FILTER = [
    # TC-6 sessions
    '2025_10_14_10_25_19_trial_1_TC', # D1 B1
    '2025_10_14_14_13_15_trial_1_TC', # D1 B2
    '2025_10_15_10_09_28_trial_1_TC', # D2 B1  
    '2025_10_15_14_16_21_trial_1_TC', # D2 B2    
    '2025_10_16_09_59_15_trial_1_TC', # D3 B1     
    '2025_10_16_14_12_17_trial_1_TC', # D3 B2    
    '2025_10_17_10_05_03_trial_1_TC', # D4 B1
    '2025_10_17_14_18_23_trial_1_TC', # D4 B2
    
   # TC-7 sessions 
    '2025_10_14_10_35_03_trial_1_TC', # D1 B1
    '2025_10_14_14_25_09_trial_1_TC', # D1 B2
    '2025_10_15_10_20_58_trial_1_TC', # D2 B1  
    '2025_10_15_14_30_16_trial_1_TC', # D2 B2  
    '2025_10_16_10_10_37_trial_1_TC', # D3 B1 
    '2025_10_16_14_28_16_trial_1_TC', # D3 B2
    '2025_10_17_10_18_17_trial_1_TC', # D4 B1
    '2025_10_17_14_30_05_trial_1_TC', # D4 B2

    # TP-3 sessions
    '2025_10_14_10_47_52_trial_1_TP', # D1 B1
    '2025_10_14_14_35_32_trial_1_TP', # D1 B2
    '2025_10_15_10_37_01_trial_1_TP', # D2 B1    
    '2025_10_15_14_43_14_trial_1_TP', # D2 B2
    '2025_10_16_10_24_06_trial_1_TP', # D3 B1
    '2025_10_16_14_48_01_trial_1_TP', # D3 B2
    '2025_10_17_10_34_57_trial_1_TP', # D4 B1
    '2025_10_17_14_45_34_trial_1_TP', # D4 B2

    # TP-4 sessions 
    '2025_10_14_11_00_58_trial_1_TP', # D1 B1
    '2025_10_14_14_49_36_trial_1_TP', # D1 B2
    '2025_10_15_10_52_34_trial_1_TP', # D2 B1
    '2025_10_15_14_57_00_trial_1_TP', # D2 B2
    '2025_10_16_10_37_16_trial_1_TP', # D3 B1
    '2025_10_16_15_00_56_trial_1_TP', # D3 B2
    '2025_10_17_10_48_37_trial_1_TP', # D4 B1
    '2025_10_17_14_54_47_trial_1_TP', # D4 B2
]

VIDEO_GROUP_FILTER = None              # Session group: 'TC', 'TP', or None for all groups
EXCLUSION_FILTER = []                  # Exclude any files/sessions containing these substrings
                                       # Uses SUBSTRING matching
                                       # Example: 'badvideo' excludes files with 'badvideo' anywhere in name
LABEL_TYPE = 'points'

# Define functions

In [3]:
def kl_gaussian_diag(A: np.ndarray, B: np.ndarray, eps: float = 1e-12) -> float:
    """
    Compute KL divergence KL(A || B) assuming diagonal (independent) Gaussians.
    
    Parameters:
    -----------
    A : np.ndarray, shape (n_samples, n_features)
        Samples from distribution A
    B : np.ndarray, shape (m_samples, n_features)
        Samples from distribution B
    eps : float
        Small value added to variances for numerical stability
    
    Returns:
    --------
    float: KL(A || B) = sum of 1D Gaussian KL divergences
    array: Per-feature KL contributions
    
    Formula for each feature:
    KL = 0.5 * [ log(var_B/var_A) + (var_A + (mean_A - mean_B)^2)/var_B - 1 ]
    """
    # Per-feature means
    mu_A = np.nanmean(A, axis=0)  # Shape: (n_features,)
    mu_B = np.nanmean(B, axis=0)
    
    # Per-feature variances (unbiased estimator with ddof=1)
    var_A = np.nanvar(A, axis=0, ddof=1) + eps    # eps = 1e-12, used to prevent div by zero
    var_B = np.nanvar(B, axis=0, ddof=1) + eps
    
    # Sum of 1D KL divergences
    kl_per_feature = 0.5 * (np.log(var_B / var_A) + (var_A + (mu_A - mu_B)**2) / var_B - 1.0)
    kl_total = np.sum(kl_per_feature)
    
    return kl_total, kl_per_feature


def kl_gaussian_full(A: np.ndarray, B: np.ndarray, eps: float = 1e-6) -> float:
    """
    Compute KL divergence KL(A || B) for full multivariate Gaussians.
    
    Parameters:
    -----------
    A : np.ndarray, shape (n_samples, n_features)
        Samples from distribution A
    B : np.ndarray, shape (m_samples, n_features)
        Samples from distribution B
    eps : float
        Small value added to diagonal of covariance for stability
    
    Returns:
    --------
    float: KL(A || B)
    
    Formula:
    KL = 0.5 * [ tr(Sigma_B^{-1} Sigma_A) + (mu_B - mu_A)^T Sigma_B^{-1} (mu_B - mu_A) 
                 - d + log(det(Sigma_B) / det(Sigma_A)) ]
    """
    n_samples, d = A.shape
    
    # Estimate means
    mu_A = np.nanmean(A, axis=0)
    mu_B = np.nanmean(B, axis=0)
    
    # Estimate covariance matrices (add eps*I for numerical stability)
    Sigma_A = np.cov(A, rowvar=False) + eps * np.eye(d)
    Sigma_B = np.cov(B, rowvar=False) + eps * np.eye(d)
    
    # Compute KL divergence terms
    inv_Sigma_B = np.linalg.inv(Sigma_B)
    diff = mu_B - mu_A
    
    trace_term = np.trace(inv_Sigma_B @ Sigma_A)
    mahalanobis_term = diff.T @ inv_Sigma_B @ diff
    logdet_term = np.log(np.linalg.det(Sigma_B) / np.linalg.det(Sigma_A))
    
    kl_div = 0.5 * (trace_term + mahalanobis_term - d + logdet_term)
    
    return kl_div


print("KL divergence functions loaded.")

KL divergence functions loaded.


## Extract the on and off frame indexes

In [4]:
def extract_CSon_CSoff_frames(Feature_vector, trial_definitions_df, feature_names, verbose=True):
    """
    Extract CS-on and CS-off frames from full feature vector.
    
    Parameters:
    -----------
    Feature_vector : np.ndarray, shape (10, n_frames)
        Full video feature array
    trial_definitions_df : pd.DataFrame
        Trial timing information
    feature_names : list
        Names of the 10 features
    verbose : bool
        If True, prints detailed extraction information. If False, runs silently.
    
    Returns:
    --------
    CS_on_samples : np.ndarray, shape (n_CS_on_frames, 10)
        CS-on data in (samples, features) format
    CS_off_samples : np.ndarray, shape (n_CS_off_frames, 10)
        CS-off data in (samples, features) format
    extraction_info : dict
        Metadata about extraction process
    """
    
    if verbose:
        print("="*30)
        print("Extracting CS-on frames (during trials)...")
    
    CS_on_frames = []
    CS_on_frame_indices = []
    
    for idx, row in trial_definitions_df.iterrows():
        trial_num = row['trial_num']
        start_frame = row['start_frame']
        end_frame = row['end_frame']
        
        # Extract frames for this trial
        trial_frames = Feature_vector[:, start_frame:end_frame+1]  # Shape: (10, num_frames_in_trial)
        CS_on_frames.append(trial_frames)
        CS_on_frame_indices.extend(range(start_frame, end_frame+1))
        
        if verbose:
            print(f"  Trial {trial_num}: frames {start_frame}-{end_frame} ({end_frame - start_frame + 1} frames)")
    
    # Concatenate all CS-on frames
    CS_on_data = np.concatenate(CS_on_frames, axis=1)  # Shape: (10, total_CS_on_frames)
    if verbose:
        print(f"Total CS-on frames: {CS_on_data.shape[1]}")
    
    # Extract CS-off frames
    if verbose:
        print("\nExtracting CS-off frames (inter-trial intervals only)...")
    
    CS_off_frames = []
    CS_off_frame_indices = []

    # If doesn't include the last off-ITI, include it!
    off_sess_num = len(trial_definitions_df) - 1
    if int(trial_definitions_df['end_frame'][len(trial_definitions_df)-1]) != len(Feature_vector[0]):
        off_sess_num = off_sess_num + 1

    # Inter-trial intervals (between consecutive trials)
    for idx in range(off_sess_num):
        if idx == (off_sess_num-1):
            interval_start = trial_definitions_df.iloc[idx]['end_frame']+1
            interval_end = len(Feature_vector[0])
        else:
            current_trial = trial_definitions_df.iloc[idx]
            next_trial = trial_definitions_df.iloc[idx + 1]
            
            interval_start = current_trial['end_frame'] + 1
            interval_end = next_trial['start_frame'] - 1
        
        if interval_end >= interval_start:
            interval_frames = Feature_vector[:, interval_start:interval_end+1]
            CS_off_frames.append(interval_frames)
            CS_off_frame_indices.extend(range(interval_start, interval_end+1))
            
            if verbose:
                #print(f"  After trial {current_trial['trial_num']}: frames {interval_start}-{interval_end} ({interval_end - interval_start + 1} frames)")
                print(f"  After trial {str(idx+1)}: frames {interval_start}-{interval_end} ({interval_end - interval_start + 1} frames)")

        else:
            if verbose:
                print(f"  After trial {str(idx+1)}: No interval (trials adjacent)") # See alt if doesnt work
    
    # Concatenate all CS-off frames
    if CS_off_frames:
        CS_off_data = np.concatenate(CS_off_frames, axis=1)  # Shape: (10, total_CS_off_frames)
        if verbose:
            print(f"Total CS-off frames: {CS_off_data.shape[1]}")
    else:
        raise ValueError("No CS-off frames found! Trials may be adjacent with no inter-trial intervals.")
    
    # Transpose to (samples, features) format
    if verbose:
        print("\nTransposing to (samples, features) format...")
    CS_on_samples = CS_on_data.T   # Shape: (n_CS_on_frames, 10)
    CS_off_samples = CS_off_data.T  # Shape: (n_CS_off_frames, 10)
    
    if verbose:
        print(f"  CS-on:  {CS_on_samples.shape}")
        print(f"  CS-off: {CS_off_samples.shape}")
    
    # NaN analysis
    if verbose:
        print("\nNaN analysis:")
    CS_on_nans = np.sum(np.isnan(CS_on_samples), axis=0)
    CS_off_nans = np.sum(np.isnan(CS_off_samples), axis=0)
    
    if verbose:
        print(f"{'Feature':<20} {'CS-on NaNs':<15} {'CS-off NaNs':<15} {'CS-on %':<10} {'CS-off %':<10}")
        print("-"*70)
        for i, fname in enumerate(feature_names):
            cs_on_pct = 100 * CS_on_nans[i] / CS_on_samples.shape[0]
            cs_off_pct = 100 * CS_off_nans[i] / CS_off_samples.shape[0]
            print(f"{fname:<20} {CS_on_nans[i]:<15} {CS_off_nans[i]:<15} {cs_on_pct:<10.2f} {cs_off_pct:<10.2f}")
    
    # Store extraction info
    extraction_info = {
        'n_CS_on_frames': CS_on_samples.shape[0],
        'n_CS_off_frames': CS_off_samples.shape[0],
        'CS_on_frame_indices': CS_on_frame_indices,
        'CS_off_frame_indices': CS_off_frame_indices,
        'CS_on_nans_per_feature': CS_on_nans.tolist(),
        'CS_off_nans_per_feature': CS_off_nans.tolist()
    }
    
    return CS_on_samples, CS_off_samples, extraction_info

print("Data extraction functions defined.")

Data extraction functions defined.


In [5]:
def compute_kl_divergence(CS_on_samples, CS_off_samples, feature_names):
    """a
    Compute KL divergence between CS-on and CS-off distributions.
    
    Parameters:
    -----------
    CS_on_samples : np.ndarray, shape (n_samples, n_features)
        CS-on data
    CS_off_samples : np.ndarray, shape (m_samples, n_features)
        CS-off data
    feature_names : list
        Names of features
    
    Returns:
    --------
    results_dict : dict
        All KL divergence results and metadata
    """
    
    print("="*30)
    print("Computing KL divergence...")
    
    # Primary: KL(CS-on || CS-off)
    kl, kl_per_feature = kl_gaussian_diag(CS_on_samples, CS_off_samples)
    print(f"\nKL(CS-on || CS-off) = {kl:.6f}")

    # Per-feature contributions
    print(f"\n{'='*30}")
    print(f"Per-feature KL contributions:")
    print(f"{'='*30}")
    print(f"{'Feature':<20} {'KL contrib':<15} {'% of total':<12}")
    print("-"*50)
    
    # Compile results
    results_dict = {
        'kl_all': float(kl),
        'kl_per_feature' : kl_per_feature
    }
    
    return results_dict


print("KL computation and results functions defined.")

KL computation and results functions defined.


## STEP 5: Define master use function.

**analyze_video_kl_divergence()** - The top-level coordinator

**Purpose:**
Orchestrates the entire KL divergence pipeline for a single video. This function:
1. Validates inputs (checks files exist)
2. Loads data (feature vector + trial definitions)
3. Calls extraction function (Block 2)
4. Calls KL computation function (Block 3)
5. **Optionally computes circshift null distribution (Block 3.5)**
6. **Calculates p-values comparing observed KL to null distribution**
7. Saves results to JSON and CSV
8. Returns formatted results

**New Parameters:**
- `compute_null` (bool): Whether to compute exhaustive circshift null distribution (~30-60s per video)
- `show_null_diagnostic` (bool): Whether to display KL vs shift diagnostic plot

**Output additions:**
- If `compute_null=True`, adds null statistics columns to CSV: null mean/std, one-tailed p-value, two-tailed p-value (for all three KL measures)
- Diagnostic plot shows whether temporal alignment matters (peak at shift=0, trough at shift~1020) or doesn't matter (flat line)

In [6]:
def _parse_start_end_from_row(row):
    import re 
    # Rejoin everything after the first column
    rest = ",".join(row[1:]).strip()
    # Find all [...] groups
    groups = re.findall(r"\[(.*?)\]", rest)
    starts, ends = [], []
    if len(groups) >= 1:
        starts = [int(x) for x in groups[0].split(",") if x.strip()]
    if len(groups) >= 2:
        ends   = [int(x) for x in groups[1].split(",") if x.strip()]
    return starts, ends

def get_session_splits(csv_path, session_name):
    import csv
    with open(csv_path, newline="") as f:
        reader = csv.reader(f)
        for row in reader:
            if not row:
                continue
            date = row[0].strip()
            if date != session_name:
                continue
            return _parse_start_end_from_row(row)
    raise ValueError(f"Session {session_name!r} not found in CSV")


def remove_poor_values(feature_vector,session):
    csv_path = "/n/holylabs/gershman_lab/Users/zkelso/video_splits.csv"
    start_frames, end_frames = get_session_splits(csv_path, session)
    
    for s, e in zip(start_frames, end_frames):
        feature_vector[:, s:e+1] = np.nan

    return feature_vector


In [7]:
def analyze_video_kl_divergence(video_name, 
                                home_dir, 
                                holylabs_dir,LABEL_TYPE,
                                save_results=True,
                                output_dir=None,
                                compute_null=False,
                                show_null_diagnostic=True):
    """
    Master function to compute KL divergence between CS-on and CS-off for a single video.
    
    Parameters:
    -----------
    video_name : str
        Video name (without _Feature_vector.npy extension)
    home_dir : str
        Path to HOME directory
    holylabs_dir : str
        Path to holylabs directory
    save_results : bool
        Whether to save results to JSON file
    output_dir : str, optional
        Where to save results (default: holylabs/KL_Results)
    compute_null : bool
        Whether to compute circshift null distribution (adds ~30-60s per video)
    show_null_diagnostic : bool
        Whether to display diagnostic plot of KL vs shift (only if compute_null=True)
    
    Returns:
    --------
    results_df : pd.DataFrame
        Summary results
    full_results : dict
        Complete results including all metadata
    """
    
    print("="*70)
    print("KL DIVERGENCE ANALYSIS")
    print("="*70)
    print(f"Video: {video_name}")
    
    # Feature names
    feature_names = ['Areas', 'Area_percentages', 'Perimeters', 'Area_perimeter_ratios',
                    'Circularities', 'Hull_areas', 'Centroidxs', 'Centroidys', 
                    'Angles', 'Concavities','PC1','PC2']
    
    # Construct paths
    features_folder = os.path.join(holylabs_dir, 'Features')
    raw_data_folder = os.path.join(home_dir, 'Raw_data')

    feature_file = os.path.join(features_folder, f"{video_name}")
    
    # Extract session prefix
    if "_regions_" in video_name:
        session_prefix = video_name.split("_regions_")[0]
    else:
        raise ValueError(f"Could not extract session prefix from: {video_name}")

    if session_prefix == "2025_10_15_14_16_21_trial_1_TC":
        trial_csv_path = "/n/holylabs/gershman_lab/Users/zkelso/Raw_data/Stuff_already_on_HL/2025_10_15_14_16_21_trial_1_TC/trial_definitions.csv"
    elif session_prefix == "2025_10_16_14_12_17_trial_1_TC":
        trial_csv_path = "/n/holylabs/gershman_lab/Users/zkelso/Raw_data/Stuff_already_on_HL/2025_10_16_14_12_17_trial_1_TC/trial_definitions.csv"
    elif session_prefix == "2025_10_17_10_05_03_trial_1_TC":
        trial_csv_path = "/n/holylabs/gershman_lab/Users/zkelso/Raw_data/Stuff_already_on_HL/2025_10_17_10_05_03_trial_1_TC/trial_definitions.csv"
    elif session_prefix == "2025_10_16_09_59_15_trial_1_TC":
        trial_csv_path = "/n/holylabs/gershman_lab/Users/zkelso/Raw_data/Stuff_already_on_HL/2025_10_16_09_59_15_trial_1_TC/trial_definitions.csv"
    elif session_prefix == "2025_10_17_14_18_23_trial_1_TC":
        trial_csv_path = "/n/holylabs/gershman_lab/Users/zkelso/Raw_data/Stuff_already_on_HL/2025_10_17_14_18_23_trial_1_TC/trial_definitions.csv"
    else:
        trial_csv_path = os.path.join(raw_data_folder, session_prefix, "trial_definitions.csv")
    
    print(f"Session: {session_prefix}")
    
    # Check files exist
    if not os.path.exists(feature_file):
        raise FileNotFoundError(f"Feature file not found: {feature_file}")
    
    if not os.path.exists(trial_csv_path):
        raise FileNotFoundError(f"Trial definitions not found: {trial_csv_path}")
    
    # Load data
    print("\nLoading data...")
    Feature_vector = np.load(feature_file)
    print(f"  Feature vector: {Feature_vector.shape}")
    
    trial_definitions_df = pd.read_csv(trial_csv_path)
    print(f"  Trial definitions: {len(trial_definitions_df)} trials")

    # Remove QC'd out portions of the feature vector
    # Feature_vector = remove_poor_values(Feature_vector,video_name.split('_fullvideo')[0])
    
    # Extract CS-on and CS-off frames
    CS_on_samples, CS_off_samples, extraction_info = extract_CSon_CSoff_frames(
        Feature_vector, trial_definitions_df, feature_names
    )
    
    # Compute KL divergence
    results_dict = compute_kl_divergence(CS_on_samples, CS_off_samples, feature_names)
    
    # Create results DataFrame
    results_df = create_results_dataframe(
        video_name, session_prefix, results_dict, feature_names, extraction_info)
    
    # Compile full results
    full_results = {
        'video': video_name,
        'session': session_prefix,
        'extraction_info': extraction_info,
        'kl_results': results_dict,
    }
    
    # Save results if requested
    if save_results:
        if output_dir is None:
            output_dir = os.path.join(holylabs_dir, 'KL_Results')
        
        os.makedirs(output_dir, exist_ok=True)
        
        # Save JSON
        json_path = os.path.join(output_dir, f"{video_name}_KL_results.json")
        with open(json_path, 'w') as f:
            json.dump(full_results, f, indent=2)
        
        print(f"\n{'='*70}")
        print(f"Results saved to: {json_path}")
        
        # Save CSV
        csv_path = os.path.join(output_dir, f"{video_name}_KL_results.csv")
        results_df.to_csv(csv_path, index=False)
        print(f"CSV saved to: {csv_path}")
    
    print(f"\n{'='*70}")
    print("Analysis complete")
    print("="*70)
    
    return results_df, full_results


def create_results_dataframe(video_name, session_prefix, results_dict, feature_names, extraction_info):
    """
    Create a DataFrame with KL divergence results.
    
    Parameters:
    -----------
    video_name : str
        Name of the video analyzed
    session_prefix : str
        Session identifier
    results_dict : dict
        KL divergence results
    extraction_info : dict
        Frame extraction metadata
    null_stats : dict, optional
        Null distribution statistics (if compute_null=True)
    
    Returns:
    --------
    pd.DataFrame : Summary results
    """
    
    # Extract worm regions (coordinates) from video name
    try:
        if "_regions_" in video_name and "_fullvideo" in video_name:
            regions_part = video_name.split("_regions_")[1].split("_fullvideo")[0]
            worm_regions = regions_part
        else:
            worm_regions = "unknown"
    except Exception as e:
        print(f"Warning: Could not extract worm_regions from video name: {e}")
        worm_regions = "unknown"
    
    # Create main results row
    main_results = {
        'video': video_name,
        'session': session_prefix,
        'Troupe':'',
        'Day':'',
        'Block':'',
        'worm_regions': worm_regions,  
        'n_CSon_frames': extraction_info['n_CS_on_frames'],
        'n_CSoff_frames': extraction_info['n_CS_off_frames'],
        'KL': results_dict['kl_all'],
        'KL_per_feature': results_dict['kl_per_feature']
    }

    # Add per-feature contributions
    for i, feature_data in enumerate(results_dict['kl_per_feature']):
        main_results[f'{feature_names[i]}_KL'] = feature_data
    
    df = pd.DataFrame([main_results])
    
    return df


print("Master function with null distribution support defined.")

Master function with null distribution support defined.


## STEP 6: Video configuration and execution

**Purpose:** Run KL divergence analysis with optional null distribution testing - just set the session and parameters, then go. This expects that feature masks have been created for each worm in a given recorded video.

**Instructions:**
1. Set SESSION to your session prefix (date/time/trial)
2. Set `COMPUTE_NULL = True` to enable circshift null testing (adds ~30-60s per video)
3. Set `SHOW_NULL_DIAGNOSTIC = True/False` to show/hide KL vs shift plots
4. Run this cell
5. Results are saved automatically

**What it does:**
- Finds all Feature_vector.npy files matching your session
- Processes each one through KL divergence analysis
- **If COMPUTE_NULL=True**: Computes exhaustive circshift null distribution and p-values
- Saves individual JSON/CSV files per video
- Creates combined CSV with all results (including null statistics if enabled)
- Prints significance summary (how many worms show p < 0.05)

In [8]:
# ============================================================================
# CONFIGURATION SECTION
# ============================================================================

HOME = "/n/holylabs/gershman_lab/Users/zkelso/"
NETSCRATCH = "/n/netscratch/gershman_lab/Lab/zkelso/"
HOLYLABS = '/n/holylabs/gershman_lab/Users/zkelso/'

# Output directories
KL_RESULTS_DIR = os.path.join(HOLYLABS, 'KL_divergence_results')
INDIVIDUAL_METADATA_DIR = os.path.join(KL_RESULTS_DIR, 'individual_video_KL_metadata')
INDIVIDUAL_SESSION_DIR = os.path.join(KL_RESULTS_DIR, 'individual_session_KL_results')

# Create directories if they don't exist
os.makedirs(KL_RESULTS_DIR, exist_ok=True)
os.makedirs(INDIVIDUAL_METADATA_DIR, exist_ok=True)
os.makedirs(INDIVIDUAL_SESSION_DIR, exist_ok=True)

SAVE_RESULTS = True

# Null distribution settings
COMPUTE_NULL = False  # Set to False to skip circshift null computation (saves time)
SHOW_NULL_DIAGNOSTIC = False  # Set to False to suppress diagnostic plots

# ============================================================================
# END CONFIGURATION SECTION
# ============================================================================







# Normalize filters to lists for uniform processing
def normalize_filter(filter_value):
    """Convert filter to list format. None/empty → None, string → [string], list → list"""
    if filter_value is None or filter_value == '' or filter_value == []:
        return None
    if isinstance(filter_value, str):
        return [filter_value]
    return filter_value

VIDEO_PREFIX_FILTER = normalize_filter(VIDEO_PREFIX_FILTER)
VIDEO_GROUP_FILTER = normalize_filter(VIDEO_GROUP_FILTER)
EXCLUSION_FILTER = normalize_filter(EXCLUSION_FILTER)


# Display filter configuration
print("="*70)
print("KL DIVERGENCE BATCH PROCESSING")
print("="*70)

# Helper functions for filtering
def should_exclude(name, exclusion_patterns):
    """
    Check if name should be excluded based on substring matching.
    Returns (should_exclude: bool, matched_pattern: str or None)
    """
    if not exclusion_patterns:
        return False, None
    
    for pattern in exclusion_patterns:
        if pattern in name:
            return True, pattern
    return False, None


def matches_prefix(name, prefix_patterns):
    """Check if name starts with ANY of the prefix patterns"""
    if not prefix_patterns:  # None = match all
        return True
    return any(name.startswith(prefix) for prefix in prefix_patterns)


def matches_group(session_name, group_patterns):
    """Check if session name ends with _{ANY} of the group patterns (TC/TP)"""
    if not group_patterns:  # None = match all
        return True
    return any(session_name.endswith(f'_{group}') for group in group_patterns)


def extract_session_from_feature_file(filename,LABEL_TYPE):
    """
    Extract session name from feature file.
    Example: '2025_10_17_14_30_05_trial_1_TC_regions_100_200_fullvideo_Feature_vector.npy'
    Returns: '2025_10_17_14_30_05_trial_1_TC'
    """
    basename = os.path.basename(filename)
    # Remove '_Feature_vector.npy' suffix
    basename = basename.replace(f"_{LABEL_TYPE}_Feature_vector.npy", '')
    # Split on '_regions_' and take first part
    if '_regions_' in basename:
        return basename.split('_regions_')[0]
    return basename


# ============================================================================
# AUTO-DISCOVER AND FILTER SESSIONS
# ============================================================================

print("\n" + "="*70)
print("DISCOVERING FEATURE FILES")
print("="*70)

# Find all FINAL feature files
features_folder = os.path.join(HOLYLABS, 'Features')
all_feature_files = glob.glob(os.path.join(features_folder, f"*_FINAL_Feature_vector.npy"))

print(f"\nFound {len(all_feature_files)} total feature files in {features_folder}")

matched_files = [path for path in all_feature_files if any(id in path for id in VIDEO_PREFIX_FILTER)]

# Check if any matches were found
if matched_files:
    print(f"Found {len(matched_files)} matching files:")
    for file in matched_files:
        print(f" - {file}")
else:
    print("No files found matching the VIDEO_PREFIX_FILTER.")

# Extract unique session names
all_sessions = []
for feature_file in matched_files:
    session = extract_session_from_feature_file(feature_file,LABEL_TYPE)
    all_sessions.append(session)

all_sessions = sorted([s for s in all_sessions if s.startswith('2025')])
print(f"Found {len(all_sessions)} unique sessions")

# Now filter feature files within each session
SESSIONS_TO_PROCESS = all_sessions

print(f"\n{'='*70}")
print(f"SESSIONS TO PROCESS: {len(SESSIONS_TO_PROCESS)}")
print(f"{'='*70}")
for idx, session in enumerate(SESSIONS_TO_PROCESS, 1):
    print(f"  {idx}. {session}")
print()


# No need to modify anything below this unless intentionally altering code.



# ============================================================================
# PROCESS EACH SESSION
# ============================================================================

all_sessions_results = []  # Store results from ALL sessions
SESSIONS_TO_PROCESS = np.unique(SESSIONS_TO_PROCESS).tolist()
for session_idx, SESSION in enumerate(SESSIONS_TO_PROCESS, 1):
    
    print("\n" + "="*70)
    print(f"SESSION {session_idx}/{len(SESSIONS_TO_PROCESS)}: {SESSION}")
    print("="*70 + "\n")
    
    # Find all feature files for current/given session
    pattern = os.path.join(HOLYLABS, 'Features', f'{SESSION}_regions_*_FINAL_Feature_vector.npy')
    feature_files = glob.glob(pattern)
        
    session_results = []  # Store results from current/given session

    # Process each video in this session
    for idx, feature_file in enumerate(feature_files, 1):
        if "FINAL" not in feature_file:
            continue
        else:
            video = os.path.basename(feature_file).replace(f'{LABEL_TYPE}_Feature_vector.npy', '')
            regions = video.split('_regions_')[1].split('_fullvideo')[0]
            
            print(f"  [{idx}/{len(feature_files)}] regions_{regions}")
            
            try:
                df, results = analyze_video_kl_divergence(
                    video_name=video,
                    home_dir=HOME,
                    holylabs_dir=HOLYLABS,
                    LABEL_TYPE= LABEL_TYPE,
                    save_results=False,  # We control saving here
                    compute_null=COMPUTE_NULL,
                    show_null_diagnostic=SHOW_NULL_DIAGNOSTIC
                )
                session_results.append(df)
                all_sessions_results.append(df)  # Also add to combined list
    
                kl = results['kl_results']['kl_all']
                print(f"      KL = {kl:.4f}")
                
                if COMPUTE_NULL and results['null_statistics'] is not None:
                    p_val = results['null_statistics']['on_given_off']['p_value_one_tailed']
                    print(f"      p-value (one-tailed) = {p_val:.4f}")
                
                # ================================================================
                # Save individual video metadata (JSON only - contains extra info)
                # ================================================================
                if SAVE_RESULTS:
                    individual_json = os.path.join(INDIVIDUAL_METADATA_DIR, f'{video}_KL_results.json')
                    
                    # Helper function to convert numpy types to native Python for JSON
                    def convert_numpy(obj):
                        if isinstance(obj, np.integer):
                            return int(obj)
                        elif isinstance(obj, np.floating):
                            return float(obj)
                        elif isinstance(obj, np.ndarray):
                            return obj.tolist()
                        elif isinstance(obj, dict):
                            return {k: convert_numpy(v) for k, v in obj.items()}
                        elif isinstance(obj, list):
                            return [convert_numpy(item) for item in obj]
                        else:
                            return obj
                    
                    # Convert results to JSON-serializable format
                    results_serializable = convert_numpy(results)
                    
                    with open(individual_json, 'w') as f:
                        json.dump(results_serializable, f, indent=2)
                    
                    print(f"      Metadata saved: {os.path.basename(individual_json)}")
                # ================================================================
                
            except Exception as e:
                print(f"      ERROR: {e}")
                traceback.print_exc()
                continue
        
    # Save per-session CSV in subdirectory
    if session_results and SAVE_RESULTS:
        session_df = pd.concat(session_results, ignore_index=True)
        
        print(f"\n  Session {SESSION} summary:")
        print(f"    Videos processed: {len(session_df)}")
        #print(f"    Mean KL: {['KL_CSon_given_CSoff'].mean():.4f}")
        
        if COMPUTE_NULL and 'p_one_tailed_CSon_given_CSoff' in session_df.columns:
            n_sig = np.sum(session_df['p_one_tailed_CSon_given_CSoff'] < 0.05)
            print(f"    Significant (p < 0.05): {n_sig}/{len(session_df)} worms")
        
        # Save per-session file in subdirectory
        session_csv = os.path.join(INDIVIDUAL_SESSION_DIR, f'{SESSION}_KL_results.csv')
        session_df.to_csv(session_csv, index=False)
        print(f"    Saved: {session_csv}")

            
            
# ============================================================================
# COMBINED RESULTS ACROSS ALL SESSIONS
# ============================================================================

if all_sessions_results:
    print("\n" + "="*70)
    print("COMBINED RESULTS - ALL SESSIONS")
    print("="*70)
    
    combined_df = pd.concat(all_sessions_results, ignore_index=True)
    
    print(f"Total videos: {len(combined_df)}")
    print(f"Total sessions: {len(SESSIONS_TO_PROCESS)}")
    #print(f"\nKL(CS-on || CS-off) across all sessions:")
    #print(f"  Mean:   {combined_df['KL_CSon_given_CSoff'].mean():.4f}")
    #print(f"  Median: {combined_df['KL_CSon_given_CSoff'].median():.4f}")
    #print(f"  Std:    {combined_df['KL_CSon_given_CSoff'].std():.4f}")
    #print(f"  Range:  {combined_df['KL_CSon_given_CSoff'].min():.4f} - {combined_df['KL_CSon_given_CSoff'].max():.4f}")
    
    if COMPUTE_NULL and 'p_one_tailed_CSon_given_CSoff' in combined_df.columns:
        print(f"\nNull hypothesis testing:")
        n_sig_05 = np.sum(combined_df['p_one_tailed_CSon_given_CSoff'] < 0.05)
        n_sig_01 = np.sum(combined_df['p_one_tailed_CSon_given_CSoff'] < 0.01)
        print(f"  Significant at p < 0.05: {n_sig_05}/{len(combined_df)} ({100*n_sig_05/len(combined_df):.1f}%)")
        print(f"  Significant at p < 0.01: {n_sig_01}/{len(combined_df)} ({100*n_sig_01/len(combined_df):.1f}%)")
    
    # Per-session breakdown
    print(f"\nPer-session summary:")
    #session_summary = combined_df.groupby('session')['KL_CSon_given_CSoff'].agg(['count', 'mean', 'std'])
    #print(session_summary)
    
    # Save master combined file with timestamp
    if SAVE_RESULTS:
        timestamp = datetime.datetime.now().strftime("%Y_%m_%d_%H_%M_%S")
        master_filename = f'Tasmanian_Conditioning_KL_Results_COMPILED_{timestamp}.csv'
        master_csv = os.path.join(KL_RESULTS_DIR, master_filename)
        combined_df.to_csv(master_csv, index=False)
        
        print(f"\n{'='*70}")
        print(f"MASTER FILE SAVED")
        print(f"{'='*70}")
        print(f"Filename: {master_filename}")
        print(f"Full path: {master_csv}")
        print(f"\nUse this file for learning curve plotting (KL vs. Day)")

else:
    print("\nNo results to combine - all sessions failed or had no videos.")

KL DIVERGENCE BATCH PROCESSING

DISCOVERING FEATURE FILES

Found 192 total feature files in /n/holylabs/gershman_lab/Users/zkelso/Features
Found 192 matching files:
 - /n/holylabs/gershman_lab/Users/zkelso/Features/2025_10_14_10_25_19_trial_1_TC_regions_1072_463_1341_1389_fullvideo_frame_split_FINAL_Feature_vector.npy
 - /n/holylabs/gershman_lab/Users/zkelso/Features/2025_10_14_10_25_19_trial_1_TC_regions_117_458_362_1397_fullvideo_frame_split_FINAL_Feature_vector.npy
 - /n/holylabs/gershman_lab/Users/zkelso/Features/2025_10_14_10_25_19_trial_1_TC_regions_1333_453_1564_1391_fullvideo_frame_split_FINAL_Feature_vector.npy
 - /n/holylabs/gershman_lab/Users/zkelso/Features/2025_10_14_10_25_19_trial_1_TC_regions_375_468_599_1397_fullvideo_frame_split_FINAL_Feature_vector.npy
 - /n/holylabs/gershman_lab/Users/zkelso/Features/2025_10_14_10_25_19_trial_1_TC_regions_604_461_854_1394_fullvideo_frame_split_FINAL_Feature_vector.npy
 - /n/holylabs/gershman_lab/Users/zkelso/Features/2025_10_14_10_25

In [9]:
print(f"\n{'='*70}")
print(f"MASTER FILE SAVED")
print(f"{'='*70}")
print(f"Filename: {master_filename}")
print(f"{master_csv}")
print(f"\nUse this file for learning curve plotting (KL vs. Day)")


MASTER FILE SAVED
Filename: Tasmanian_Conditioning_KL_Results_COMPILED_2026_03_25_14_36_31.csv
/n/holylabs/gershman_lab/Users/zkelso/KL_divergence_results/Tasmanian_Conditioning_KL_Results_COMPILED_2026_03_25_14_36_31.csv

Use this file for learning curve plotting (KL vs. Day)
